# Intrinsics Add Thermal (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-06  
**Last Modified:** 2026-04-06  
**Status:** Draft  
**Keywords:** intrinsics, thermal, EFD, ConsDB, Zernike  

## Description

Augment the intrinsic Zernike fit table with thermal information from the EFD and ConsDB.

Key functionality:
1. Read intrinsic_fits parquet table
2. Query ConsDB for observation times (obs_start, obs_end) per visit
3. Query EFD for ESS temperatures and M1M3 thermal gradients
4. Compute delta-T quantities and merge into the table
5. Attempt to retrieve TMA truss temperatures from the EFD

**Output:** Augmented parquet table with thermal columns  
**Based on:** nightly_tablemaker.ipynb EFD/ConsDB patterns

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-06 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [EFD Thermal Queries](#thermal)
6. [TMA Truss Temperatures](#truss)
7. [Output](#output)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

input_dir = "output/fam_danish_bin1_OCS_wep_v16_8_0_dviz_v3_5_0_20260315_20260317"
input_file = "fam_danish_bin1_intrinsic_fits_wep_v16_8_0_dviz_v3_5_0_20260315_20260317.parquet"
output_file = "fam_danish_bin1_intrinsic_fits_thermal_wep_v16_8_0_dviz_v3_5_0_20260315_20260317.parquet"

consdb_url = "http://consdb-pq.consdb:8080/consdb"

# Time window for EFD temperature queries (seconds beyond obs_end)
temp_time_window_sec = 0.2

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

from astropy.time import Time, TimeDelta

from lsst.summit.utils import ConsDbClient
from lsst.summit.utils.efdUtils import makeEfdClient

from lsst.ts.xml.tables.m1m3 import *
from lsst.ts.m1m3.utils import ThermocoupleAnalysis

if 'no_proxy' in os.environ:
    os.environ['no_proxy'] += ',.consdb'
else:
    os.environ['no_proxy'] = '.consdb'

temp_time_window = TimeDelta(temp_time_window_sec, format="sec")

<a id='functions'></a>
## Helper Functions

In [ ]:
async def async_getEfdData(client, topic, obs_start, obs_end, columns=None,
                           prePadding=0, postPadding=0, index=None):
    """Async EFD query using obs_start/obs_end times.

    Adapted from async_getEfdData in intrinsics_mktable (which uses
    Butler expRecord timespans) to work with ConsDB-provided times.

    Parameters
    ----------
    client : EfdClient
        EFD client instance.
    topic : str
        EFD topic name.
    obs_start : str
        Observation start time (TAI isot string).
    obs_end : str
        Observation end time (TAI isot string).
    columns : list of str, optional
        Columns to retrieve. None for all columns.
    prePadding : float, optional
        Seconds to subtract from begin time.
    postPadding : float, optional
        Seconds to add to end time.
    index : int, optional
        SAL index for the topic.

    Returns
    -------
    data : pandas.DataFrame or None
        EFD data, or None if no data found.
    """
    begin = Time(obs_start, scale="tai")
    end = Time(obs_end, scale="tai")
    if prePadding:
        begin -= TimeDelta(prePadding, format="sec")
    if postPadding:
        end += TimeDelta(postPadding, format="sec")

    kwargs = {}
    if index is not None:
        kwargs["index"] = index
        kwargs["convert_influx_index"] = True

    data = await client.select_time_series(
        topic, columns or [], begin.utc, end.utc, **kwargs
    )
    if data is None or len(data) == 0:
        return None
    return data


async def get_ess_temperature(efd_client, obs_start, obs_end, index,
                              field="temperatureItem0"):
    """Query a single ESS temperature from the EFD.

    Parameters
    ----------
    efd_client : EfdClient
        EFD client.
    obs_start : str
        Observation start time (TAI).
    obs_end : str
        Observation end time (TAI).
    index : int
        ESS sensor index (111=cam, 112=M2, 113=M1M3, 301=outside, 122=TMA).
    field : str
        Temperature field name (default: temperatureItem0).

    Returns
    -------
    float
        Mean temperature during observation, or NaN.
    """
    data = await async_getEfdData(
        efd_client,
        "lsst.sal.ESS.temperature",
        obs_start, obs_end,
        columns=[field],
        postPadding=temp_time_window_sec,
        index=index,
    )
    if data is not None and field in data.columns:
        return data[field].mean()
    return np.nan


async def get_m1m3_gradients(efd_client, visit_table):
    """Get M1M3 thermal gradients interpolated to observation times.

    Parameters
    ----------
    efd_client : EfdClient
        EFD client.
    visit_table : DataFrame
        Must contain obs_start column (TAI isot strings).

    Returns
    -------
    DataFrame
        Columns: x_gradient, y_gradient, z_gradient, radial_gradient.
    """
    date_strings = Time(
        [str(x) for x in visit_table["obs_start"].values],
        format="isot", scale="tai"
    ).utc.isot
    data_times = pd.to_datetime(date_strings, format="ISO8601", utc=True)
    sorted_data_times = data_times.sort_values()
    start = Time(sorted_data_times[0])
    end = Time(sorted_data_times[-1])
    data_times_int = data_times.astype("int64")

    thermocouples = ThermocoupleAnalysis(efd_client)
    await thermocouples.load(start, end, time_bin=30)
    gradients = thermocouples.xyz_r_gradients
    grad_times = pd.to_datetime(
        gradients.index, format="ISO8601", utc=True
    ).astype("int64")
    t0 = grad_times[0]
    grad_times = (grad_times - t0) / 1e9
    data_times_norm = (data_times_int - t0) / 1e9

    result = {}
    for name in ["x_gradient", "y_gradient", "z_gradient", "radial_gradient"]:
        values = gradients[name].values
        val_interpolated = pd.Series(values).interpolate().values
        result[name] = np.interp(data_times_norm, grad_times, val_interpolated)

    return pd.DataFrame(result, index=visit_table.index)

<a id='data'></a>
## Data Access

In [ ]:
input_path = Path(input_dir) / input_file
df = pd.read_parquet(input_path)
print(f"Loaded {len(df)} rows from {input_path.name}")
print(f"day_obs values: {sorted(df['day_obs'].unique())}")
print(f"Columns: {len(df.columns)}")

### Get observation times from ConsDB

In [ ]:
cdb_client = ConsDbClient(consdb_url)

# Build ConsDB query for obs_start and obs_end for all visits
day_obs_list = sorted(df["day_obs"].unique())
day_obs_str = ", ".join(str(d) for d in day_obs_list)

query = f"""
    SELECT
        e.day_obs AS day_obs,
        e.seq_num AS seq_num,
        e.obs_start AS obs_start,
        e.obs_end AS obs_end
    FROM cdb_lsstcam.exposure e
    WHERE e.day_obs IN ({day_obs_str})
    ORDER BY e.day_obs, e.seq_num
"""
consdb_df = cdb_client.query(query).to_pandas()
print(f"ConsDB returned {len(consdb_df)} exposures")

# Merge obs_start and obs_end into our table
df = df.merge(
    consdb_df[["day_obs", "seq_num", "obs_start", "obs_end"]],
    on=["day_obs", "seq_num"],
    how="left",
)
n_matched = df["obs_start"].notna().sum()
print(f"Matched obs times for {n_matched}/{len(df)} rows")

<a id='thermal'></a>
## EFD Thermal Queries

In [ ]:
efd_client = makeEfdClient()

### ESS air temperatures

In [ ]:
# ESS temperature sensor indices
ess_sensors = {
    "cam_air_temp": 111,
    "m2_air_temp": 112,
    "m1m3_air_temp": 113,
    "outside_temp": 301,
}

# Get unique visits with valid obs times
valid = df.dropna(subset=["obs_start", "obs_end"]).copy()
unique_visits = valid[["day_obs", "seq_num", "obs_start", "obs_end"]].drop_duplicates()
print(f"Querying EFD for {len(unique_visits)} unique visits")

# Query temperatures per visit
temp_records = []
for _, row in tqdm(unique_visits.iterrows(), total=len(unique_visits)):
    record = {"day_obs": row["day_obs"], "seq_num": row["seq_num"]}
    for name, index in ess_sensors.items():
        record[name] = await get_ess_temperature(
            efd_client, row["obs_start"], row["obs_end"], index
        )
    temp_records.append(record)

temp_df = pd.DataFrame(temp_records)
print(f"Retrieved temperatures for {len(temp_df)} visits")

# Compute delta-T quantities
temp_df["m2_delta_t"] = temp_df["m2_air_temp"] - temp_df["m1m3_air_temp"]
temp_df["cam_m1m3_delta_t"] = temp_df["cam_air_temp"] - temp_df["m1m3_air_temp"]
temp_df["dome_delta_t"] = temp_df["outside_temp"] - temp_df["m1m3_air_temp"]

# Report
for col in ess_sensors:
    n_valid = temp_df[col].notna().sum()
    print(f"  {col}: {n_valid}/{len(temp_df)} valid")

### M1M3 thermal gradients

In [ ]:
# Get M1M3 thermal gradients — process one day_obs at a time to avoid
# large EFD queries that trigger connection resets.
gradient_parts = []
for day_obs_val in sorted(unique_visits["day_obs"].unique()):
    day_visits = unique_visits[unique_visits["day_obs"] == day_obs_val].reset_index(drop=True)
    print(f"  day_obs {day_obs_val}: {len(day_visits)} visits")
    try:
        gdf = await get_m1m3_gradients(efd_client, day_visits)
        gdf["day_obs"] = day_visits["day_obs"].values
        gdf["seq_num"] = day_visits["seq_num"].values
        gradient_parts.append(gdf)
    except Exception as e:
        print(f"    WARNING: failed for {day_obs_val}: {e}")
        # Fill with NaNs for this day
        gdf = pd.DataFrame({
            "x_gradient": np.nan, "y_gradient": np.nan,
            "z_gradient": np.nan, "radial_gradient": np.nan,
            "day_obs": day_visits["day_obs"].values,
            "seq_num": day_visits["seq_num"].values,
        })
        gradient_parts.append(gdf)

gradient_df = pd.concat(gradient_parts, ignore_index=True)

for col in ["x_gradient", "y_gradient", "z_gradient", "radial_gradient"]:
    n_valid = gradient_df[col].notna().sum()
    print(f"  {col}: {n_valid}/{len(gradient_df)} valid")

<a id='truss'></a>
## TMA Truss Temperatures

Two truss temperature sensors from ESS index 122:
- `temperatureItem6`: TMA Truss Temp at +X+Y
- `temperatureItem7`: TMA Truss Temp at -X-Y

In [ ]:
# TMA truss temperatures: ESS index 122, temperatureItem6 (+X+Y) and temperatureItem7 (-X-Y)
truss_sensors = {
    "tma_truss_temp_pxpy": ("temperatureItem6", 122),  # +X+Y
    "tma_truss_temp_mxmy": ("temperatureItem7", 122),  # -X-Y
}

truss_records = []
for _, row in tqdm(unique_visits.iterrows(), total=len(unique_visits)):
    record = {"day_obs": row["day_obs"], "seq_num": row["seq_num"]}
    for name, (field, index) in truss_sensors.items():
        record[name] = await get_ess_temperature(
            efd_client, row["obs_start"], row["obs_end"], index, field=field
        )
    truss_records.append(record)

truss_df = pd.DataFrame(truss_records)
for col in truss_sensors:
    n_valid = truss_df[col].notna().sum()
    print(f"  {col}: {n_valid}/{len(truss_df)} valid")

<a id='output'></a>
## Output

In [ ]:
# Merge thermal data into the intrinsic_fits table
df = df.merge(temp_df, on=["day_obs", "seq_num"], how="left")
df = df.merge(gradient_df, on=["day_obs", "seq_num"], how="left")
df = df.merge(truss_df, on=["day_obs", "seq_num"], how="left")

# Drop the obs_start/obs_end helper columns
df = df.drop(columns=["obs_start", "obs_end"], errors="ignore")

thermal_cols = (
    list(ess_sensors.keys())
    + ["m2_delta_t", "cam_m1m3_delta_t", "dome_delta_t"]
    + ["x_gradient", "y_gradient", "z_gradient", "radial_gradient"]
    + list(truss_sensors.keys())
)
print(f"Added {len(thermal_cols)} thermal columns:")
for col in thermal_cols:
    n_valid = df[col].notna().sum()
    print(f"  {col}: {n_valid}/{len(df)} valid")

In [ ]:
output_path = Path(input_dir) / output_file
df.to_parquet(output_path, index=False)
print(f"Saved {len(df)} rows x {len(df.columns)} columns to {output_path.name}")